# RoyalTDN — Fase 2: Backtesting Avanzado

## Walk-Forward Analysis con VectorBT

**Documento de referencia:** `05_ruta_implementacion_fase2_fase3.md` — Sección 5.2.1

**Objetivo:** Validar si la estrategia SMA Crossover tiene valor real o es puro ruido,
usando Walk-Forward Analysis (WFA) con métricas out-of-sample.

**Principios (doc 02):**
- Walk-forward analysis para evitar sobreoptimización.
- Métricas out-of-sample como única verdad.
- Robustez paramétrica (pequeños cambios no deben destruir el rendimiento).
- Validación cross-asset.

In [ ]:
# Celda 1: Imports
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import vectorbt as vbt

warnings.filterwarnings('ignore')
print(f"VectorBT version: {vbt.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Celda 2: Descargar datos SPY 2015-2024
print("Descargando datos SPY 2015-2024...")
spy = yf.download('SPY', start='2015-01-01', end='2024-12-31', auto_adjust=True)
close = spy['Close']
print(f"Filas: {len(close)} | Desde: {close.index[0].date()}  Hasta: {close.index[-1].date()}")
print(f"Precio rango: ${close.min():.2f} - ${close.max():.2f}")

# Plot rápido
close.plot(figsize=(14, 5), title='SPY — Precio de Cierre (2015-2024)', grid=True);

In [ ]:
# Celda 3: Definir grilla de parámetros
fast_periods = np.arange(2, 30, 2)    # SMA rápida: 2, 4, 6, ..., 28
slow_periods = np.arange(30, 200, 10) # SMA lenta: 30, 40, 50, ..., 190

print(f"Fast SMAs: {len(fast_periods)} combinaciones ({fast_periods[0]}-{fast_periods[-1]})")
print(f"Slow SMAs: {len(slow_periods)} combinaciones ({slow_periods[0]}-{slow_periods[-1]})")
print(f"Total: {len(fast_periods) * len(slow_periods)} combinaciones")

In [ ]:
# Celda 4: Calcular SMAs vectorizadas para todas las combinaciones
print("Calculando SMAs para todas las combinaciones...")

# vbt.MA.run acepta un array de ventanas y calcula todas en paralelo
fast_ma = vbt.MA.run(close, window=fast_periods)
slow_ma = vbt.MA.run(close, window=slow_periods)

print(f"Fast MA shape: {fast_ma.ma.shape}")
print(f"Slow MA shape: {slow_ma.ma.shape}")

In [ ]:
# Celda 5: Generar señales de entrada y salida
print("Generando señales de cruce...")

# fast_ma.ma_crossed_above(slow_ma) → matriz (time x param_combo)
# Cada combinación (fast_i, slow_j) produce una serie de señales
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

print(f"Entries shape: {entries.shape}")
print(f"Exits shape: {exits.shape}")
print(f"Total señales de entrada: {entries.sum().item()}")

In [ ]:
# Celda 6: Walk-Forward Analysis Manual
print("=" * 60)
print("WALK-FORWARD ANALYSIS")
print("=" * 60)

# Configuración del WFA
n_splits = 5
window_years = 3  # 3 años de entrenamiento
test_years = 1    # 1 año de prueba

# Dividir el timeline
total_years = (close.index[-1] - close.index[0]).days / 365.25
window_days = int(window_years * 365.25)
test_days = int(test_years * 365.25)

results = []

for fold in range(n_splits):
    # Definir ventanas
    train_start = close.index[0] + pd.Timedelta(days=fold * test_days)
    train_end = train_start + pd.Timedelta(days=window_days)
    test_end = train_end + pd.Timedelta(days=test_days)

    if test_end > close.index[-1]:
        print(f"Fold {fold+1}: saltando (fuera de rango)")
        continue

    # Datos de entrenamiento
    train_mask = (close.index >= train_start) & (close.index < train_end)
    test_mask = (close.index >= train_end) & (close.index < test_end)

    close_train = close[train_mask]
    close_test = close[test_mask]

    if len(close_train) < 100 or len(close_test) < 50:
        continue

    # Calcular señales en training
    fast_train = vbt.MA.run(close_train, window=fast_periods)
    slow_train = vbt.MA.run(close_train, window=slow_periods)
    entries_train = fast_train.ma_crossed_above(slow_train)
    exits_train = fast_train.ma_crossed_below(slow_train)

    # Portfolio en training
    pf_train = vbt.Portfolio.from_signals(close_train, entries_train, exits_train,
                                          freq='D', init_cash=100_000)

    # Seleccionar mejor combinación por Sharpe en training
    sharpe_train = pf_train.sharpe_ratio()
    best_idx = sharpe_train.idxmax()
    best_sharpe_train = sharpe_train.max()

    # Reconstruir best_idx como índices de fast/slow
    best_fast = fast_periods[best_idx[0]]
    best_slow = slow_periods[best_idx[1]]

    # Probar en out-of-sample
    fast_test = vbt.MA.run(close_test, window=int(best_fast))
    slow_test = vbt.MA.run(close_test, window=int(best_slow))
    entries_test = fast_test.ma_crossed_above(slow_test)
    exits_test = fast_test.ma_crossed_below(slow_test)

    pf_test = vbt.Portfolio.from_signals(close_test, entries_test, exits_test,
                                         freq='D', init_cash=100_000)

    # Métricas OOS
    oos_sharpe = pf_test.sharpe_ratio()
    oos_mdd = pf_test.max_drawdown()
    profit_factor = pf_test.trades.profit_factor() if hasattr(pf_test.trades, 'profit_factor') else 0
    total_return = pf_test.total_return()
    n_trades = pf_test.trades.count()

    results.append({
        'Fold': fold + 1,
        'Periodo Train': f"{train_start.date()} a {train_end.date()}",
        'Periodo Test': f"{train_end.date()} a {test_end.date()}",
        'Best SMA Fast': int(best_fast),
        'Best SMA Slow': int(best_slow),
        'Sharpe IS': round(float(best_sharpe_train), 2),
        'Sharpe OOS': round(float(oos_sharpe), 2),
        'MaxDD OOS': f"{float(oos_mdd)*100:.1f}%",
        'Return OOS': f"{float(total_return)*100:.1f}%",
        'Trades': int(n_trades),
    })

    print(f"Fold {fold+1}: SMA({int(best_fast)},{int(best_slow)}) | "
          f"Sharpe IS: {best_sharpe_train:.2f} → Sharpe OOS: {oos_sharpe:.2f} | "
          f"Return: {float(total_return)*100:.1f}% | DD: {float(oos_mdd)*100:.1f}%")

# Mostrar tabla resumen
df_results = pd.DataFrame(results)
print("\n" + "=" * 60)
print("RESUMEN WALK-FORWARD")
print("=" * 60)
display(df_results)

# Métricas agregadas
oos_sharpes = df_results['Sharpe OOS'].values
print(f"\n📊 Sharpe OOS promedio: {np.mean(oos_sharpes):.2f}")
print(f"📊 Sharpe OOS mediana:  {np.median(oos_sharpes):.2f}")
print(f"📊 Sharpe OOS > 1:      {sum(1 for s in oos_sharpes if s > 1)}/{len(oos_sharpes)} folds")

In [ ]:
# Celda 7: Portfolio stats y gráficos del mejor fold
print("=" * 60)
print("ANÁLISIS DEL MEJOR FOLD")
print("=" * 60)

# Encontrar el mejor fold por Sharpe OOS
if len(results) > 0:
    best_fold_row = df_results.loc[df_results['Sharpe OOS'].idxmax()]
    print(f"Mejor fold: #{int(best_fold_row['Fold'])}")
    print(f"Parámetros: SMA({int(best_fold_row['Best SMA Fast'])},{int(best_fold_row['Best SMA Slow'])})")
    print(f"Sharpe OOS: {best_fold_row['Sharpe OOS']}")
    print(f"MaxDD OOS:  {best_fold_row['MaxDD OOS']}")

    # Reconstruir el mejor fold
    best_f = int(best_fold_row['Best SMA Fast'])
    best_s = int(best_fold_row['Best SMA Slow'])

    fast_ma_full = vbt.MA.run(close, window=best_f)
    slow_ma_full = vbt.MA.run(close, window=best_s)
    entries_full = fast_ma_full.ma_crossed_above(slow_ma_full)
    exits_full = fast_ma_full.ma_crossed_below(slow_ma_full)

    pf_full = vbt.Portfolio.from_signals(close, entries_full, exits_full,
                                         freq='D', init_cash=100_000)

    print("\n📈 Equity curve (full period, mejor parámetros):")
    pf_full.plot().show()

    print("\n📊 Estadísticas del portfolio:")
    stats = pf_full.stats()
    for idx, val in stats.items():
        print(f"  {idx}: {val}")

In [ ]:
# Celda 8: Análisis QuantStats
import quantstats as qs

print("Generando reporte QuantStats...")

# Crear directorio reports
import os
os.makedirs('reports', exist_ok=True)

# Usar el mejor fold para generar returns
if len(results) > 0:
    best_fold_row = df_results.loc[df_results['Sharpe OOS'].idxmax()]
    best_f = int(best_fold_row['Best SMA Fast'])
    best_s = int(best_fold_row['Best SMA Slow'])

    fast_ma_full = vbt.MA.run(close, window=best_f)
    slow_ma_full = vbt.MA.run(close, window=best_s)
    entries_full = fast_ma_full.ma_crossed_above(slow_ma_full)
    exits_full = fast_ma_full.ma_crossed_below(slow_ma_full)

    pf_full = vbt.Portfolio.from_signals(close, entries_full, exits_full,
                                         freq='D', init_cash=100_000)

    returns = pf_full.returns()

    # Benchmark: SPY buy & hold
    spy_returns = close.pct_change().dropna()

    # Generar reporte HTML
    qs.reports.html(returns, benchmark=spy_returns,
                    output='reports/sma_wfa_report.html',
                    title='RoyalTDN — SMA Crossover WFA')
    print("✅ Reporte generado: reports/sma_wfa_report.html")
else:
    print("❌ No hay resultados para generar reporte")

---
## Prueba de Robustez

Según el documento 05, sección 5.2.2: pequeñas variaciones en los parámetros
no deben destruir el rendimiento.

In [ ]:
# Celda 9: Prueba de robustez paramétrica
print("=" * 60)
print("PRUEBA DE ROBUSTEZ — Variación paramétrica")
print("=" * 60)

# Probar variaciones alrededor del mejor par
if len(results) > 0:
    best_fold_row = df_results.loc[df_results['Sharpe OOS'].idxmax()]
    base_f = int(best_fold_row['Best SMA Fast'])
    base_s = int(best_fold_row['Best SMA Slow'])
else:
    base_f, base_s = 10, 50  # fallback

variants = [
    (base_f, base_s, 'Original (WFA)'),
    (max(2, base_f - 2), base_s, 'Fast-2'),
    (base_f + 2, base_s, 'Fast+2'),
    (base_f, max(30, base_s - 10), 'Slow-10'),
    (base_f, base_s + 10, 'Slow+10'),
    (8, 45, 'Alternativo 1'),
    (12, 55, 'Alternativo 2'),
]

robustness = []
for f, s, label in variants:
    fast_v = vbt.MA.run(close, window=f)
    slow_v = vbt.MA.run(close, window=s)
    entries_v = fast_v.ma_crossed_above(slow_v)
    exits_v = fast_v.ma_crossed_below(slow_v)
    pf_v = vbt.Portfolio.from_signals(close, entries_v, exits_v,
                                       freq='D', init_cash=100_000)
    robustness.append({
        'Variante': label,
        'SMA': f'({f},{s})',
        'Sharpe': round(float(pf_v.sharpe_ratio()), 2),
        'MaxDD': f"{float(pf_v.max_drawdown())*100:.1f}%",
        'Return': f"{float(pf_v.total_return())*100:.1f}%",
        'Trades': int(pf_v.trades.count()),
    })

df_robust = pd.DataFrame(robustness)
display(df_robust)

# Conclusión de robustez
sharpes = [r['Sharpe'] for r in robustness]
mean_sharpe = np.mean([s for s in sharpes if s != -np.inf and s != np.inf])
std_sharpe = np.std([s for s in sharpes if s != -np.inf and s != np.inf])
print(f"\n📊 Sharpe promedio: {mean_sharpe:.2f}")
print(f"📊 Sharpe std dev:  {std_sharpe:.2f}")
print(f"📊 Coef. variación: {std_sharpe/abs(mean_sharpe)*100:.1f}%" if mean_sharpe != 0 else "Sharpe=0, CV no calculable")

if std_sharpe < 0.5:
    print("✅ ROBUSTEZ: Las variaciones no destruyen el rendimiento")
else:
    print("⚠️ ROBUSTEZ: Alta sensibilidad a cambios de parámetros")

In [ ]:
# Celda 10: Validación cross-asset (QQQ e IWM)
print("=" * 60)
print("VALIDACIÓN CROSS-ASSET")
print("=" * 60)

assets = {'QQQ': 'NASDAQ Tech', 'IWM': 'Small Caps'}

if len(results) > 0:
    best_fold_row = df_results.loc[df_results['Sharpe OOS'].idxmax()]
    base_f = int(best_fold_row['Best SMA Fast'])
    base_s = int(best_fold_row['Best SMA Slow'])
else:
    base_f, base_s = 10, 50

cross_results = []
for ticker, name in assets.items():
    print(f"\nProbando {ticker} ({name})...")
    try:
        data = yf.download(ticker, start='2015-01-01', end='2024-12-31', auto_adjust=True, progress=False)
        close_a = data['Close']

        fast_a = vbt.MA.run(close_a, window=base_f)
        slow_a = vbt.MA.run(close_a, window=base_s)
        entries_a = fast_a.ma_crossed_above(slow_a)
        exits_a = fast_a.ma_crossed_below(slow_a)
        pf_a = vbt.Portfolio.from_signals(close_a, entries_a, exits_a,
                                           freq='D', init_cash=100_000)

        cross_results.append({
            'Asset': ticker,
            'Nombre': name,
            'Sharpe': round(float(pf_a.sharpe_ratio()), 2),
            'MaxDD': f"{float(pf_a.max_drawdown())*100:.1f}%",
            'Return': f"{float(pf_a.total_return())*100:.1f}%",
            'Trades': int(pf_a.trades.count()),
        })
    except Exception as e:
        print(f"  Error con {ticker}: {e}")

df_cross = pd.DataFrame(cross_results)
print("\n" + "=" * 60)
display(df_cross)

if len(cross_results) > 0:
    cross_sharpes = [r['Sharpe'] for r in cross_results]
    positive = sum(1 for s in cross_sharpes if s > 0)
    print(f"\n📊 Assets con Sharpe > 0: {positive}/{len(cross_results)}")
    if positive >= len(cross_results) * 0.5:
        print("✅ La estrategia funciona en múltiples activos")
    else:
        print("⚠️ La estrategia NO es consistente entre activos")

In [ ]:
# Celda 11: Interpretación final
print("=" * 60)
print("CONCLUSIÓN — ¿SMA CROSSOVER TIENE VALOR?")
print("=" * 60)
print("""
Criterios de evaluación (doc 02):
  1. Sharpe OOS > 1.0
  2. MDD < 30%
  3. Profit Factor > 1.5
  4. Robustez paramétrica (CV < 50%)
  5. Consistencia cross-asset
""")

if len(results) > 0:
    avg_oos = df_results['Sharpe OOS'].mean()
    print(f"\n📊 Sharpe OOS promedio: {avg_oos:.2f}")

    if avg_oos > 1.0:
        print("✅ Pasa criterio 1: Sharpe OOS > 1.0")
    else:
        print("❌ Falla criterio 1: Sharpe OOS < 1.0 — alfa cuestionable")

    print("\n⚠️ NOTA IMPORTANTE:")
    print("  El SMA crossover es la estrategia más básica posible.")
    print("  Si NO pasa estos filtros, no significa que el proyecto fracase —")
    print("  significa que necesitamos estrategias más sofisticadas en Fase 4.")
    print("  El valor de este análisis es establecer la LÍNEA DE BASE (baseline).")